In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

# Load the all details csv 
par_details = pd.read_csv(base_dir + "/Stat_csv_files/CWNS_CWS_all_details.csv") # Change to required destination

In [ ]:

# ============================================================
# DECISION TREE: Nested + Repeated Stratified CV
# Outputs:
#  - decision_tree_foldwise.csv (per outer fold metrics)
#  - decision_tree_summary.csv (mean, sd, min, max, count)
#  - decision_tree_importances.csv (mean importance per feature set)
#  - figures: mean±sd bars + boxplots
# ============================================================

# -----------------------------
# Work on a COPY (do not modify original par_details)
# -----------------------------
df = par_details.copy(deep=True)

# -----------------------------
# Target (CWS=1, CWNS=0)
# -----------------------------
y = df["Group"].map({"CWS": 1, "CWNS": 0})

# Map Sex (kept only for baseline comparison set)
df["Sex"] = df["Sex"].map({"M": 1, "F": 0})

# -----------------------------
# Feature sets
# -----------------------------
atas_features = [
    "Pause_Duration_s", "Pause_Events",
    "Mean Speech_s", "long_p_count", "short_p_count", "Speech_Rate"
]
full_features = ["Age", "Sex"] + atas_features
demo_features = ["Age", "Sex"]
atas_only_features = atas_features
age_plus_atas_features = ["Age"] + atas_features

feature_sets = {
    "Age + Sex + ATAS": full_features,
    "Age + Sex": demo_features,     # baseline
    "ATAS": atas_only_features,
    "Age + ATAS": age_plus_atas_features,
}

# -----------------------------
# CV configs
# -----------------------------
OUTER_SPLITS  = 5
OUTER_REPEATS = 5
INNER_SPLITS  = 3
SEED = 42

outer_cv = RepeatedStratifiedKFold(
    n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED
)
inner_cv = StratifiedKFold(
    n_splits=INNER_SPLITS, shuffle=True, random_state=SEED
)

# -----------------------------
# Plot helpers
# -----------------------------
def plot_metric_visuals(foldwise_df: pd.DataFrame, title_prefix: str, out_prefix: str):
    """
    Creates paper-friendly visuals:
      - mean±SD bar for Accuracy + F1
      - boxplot distribution for Accuracy + F1
    Saves figures to disk.
    """
    # Keep feature set order as they appear in the input dataframe
    feature_order = foldwise_df["Feature_Set"].drop_duplicates().tolist()

    summary = (
        foldwise_df.groupby("Feature_Set")[["Accuracy", "F1"]]
        .agg(["mean", "std", "min", "max", "count"])
        .reindex(feature_order)
    )

    labels = summary.index.tolist()

    # mean ± SD bars
    for metric in ["Accuracy", "F1"]:
        means = summary[(metric, "mean")].values
        stds  = summary[(metric, "std")].values

        plt.figure()
        plt.bar(labels, means, yerr=stds, capsize=6)
        plt.xticks(rotation=25, ha="right")
        plt.ylabel(metric)
        plt.title(f"{title_prefix} {metric} (mean ± SD) | Outer CV")
        plt.tight_layout()
        plt.savefig(f"{out_prefix}_{metric.lower()}_mean_sd.png", dpi=300)
        plt.show()

    # boxplots
    for metric in ["Accuracy", "F1"]:
        plt.figure()
        data = [foldwise_df.loc[foldwise_df["Feature_Set"] == s, metric].values for s in labels]
        plt.boxplot(data, labels=labels)
        plt.xticks(rotation=25, ha="right")
        plt.ylabel(metric)
        plt.title(f"{title_prefix} Fold-wise {metric} Distribution | Outer CV")
        plt.tight_layout()
        plt.savefig(f"{out_prefix}_{metric.lower()}_boxplot.png", dpi=300)
        plt.show()

# -----------------------------
# Randomized search space
# -----------------------------
param_distributions = {
    "max_depth": [2, 3, 4, 5, 7, None],
    "min_samples_split": [2, 4, 6, 8],
    "min_samples_leaf": [1, 2, 4],
}
N_ITER = 20

all_foldwise = []
all_importances = []  # long-format table for importances per fold

for set_name, feats in feature_sets.items():
    X = df[feats].copy()

    # Defensive numeric coercion + complete-case mask per feature set
    X = X.apply(pd.to_numeric, errors="coerce")
    mask = X.notna().all(axis=1) & y.notna()
    X = X.loc[mask]
    y_used = y.loc[mask]

    # Collect importances per outer fold to later average
    outer_importances = []

    # Track repeat/fold separately for cleaner reporting
    for outer_i, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y_used), start=1):
        repeat_id = (outer_i - 1) // OUTER_SPLITS + 1
        fold_id   = (outer_i - 1) % OUTER_SPLITS + 1

        X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
        y_train, y_test = y_used.iloc[tr_idx], y_used.iloc[te_idx]

        base = DecisionTreeClassifier(
            random_state=SEED,
            class_weight="balanced"
        )

        rs = RandomizedSearchCV(
            estimator=base,
            param_distributions=param_distributions,
            n_iter=N_ITER,
            scoring="f1",
            cv=inner_cv,
            random_state=SEED,
            n_jobs=-1
        )
        rs.fit(X_train, y_train)

        best_model = rs.best_estimator_
        y_pred = best_model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred)

        # Foldwise metrics
        all_foldwise.append({
            "Model": "DecisionTree",
            "Feature_Set": set_name,
            "Outer_Repeat": repeat_id,
            "Outer_Fold": fold_id,
            "Outer_Index": outer_i,  # 1..(splits*repeats)
            "Accuracy": float(acc),
            "F1": float(f1),
            "Best_Params": str(rs.best_params_),
        })

        # Foldwise importances
        imp = best_model.feature_importances_
        outer_importances.append(imp)

        for feat, val in zip(feats, imp):
            all_importances.append({
                "Model": "DecisionTree",
                "Feature_Set": set_name,
                "Outer_Repeat": repeat_id,
                "Outer_Fold": fold_id,
                "Outer_Index": outer_i,
                "Feature": feat,
                "Importance": float(val),
            })

# DataFrames
foldwise_df = pd.DataFrame(all_foldwise)
importances_df = pd.DataFrame(all_importances)

# Summary with true ranges (min/max)
summary_df = (
    foldwise_df.groupby(["Model", "Feature_Set"])[["Accuracy", "F1"]]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print("\n===== Decision Tree Summary (mean, SD, min, max, n) =====")
print(summary_df)

# Mean importance per feature set (averaged across outer folds)
imp_mean_df = (
    importances_df.groupby(["Model", "Feature_Set", "Feature"])["Importance"]
    .mean()
    .reset_index()
    .sort_values(["Feature_Set", "Importance"], ascending=[True, False])
)

print("\n===== Decision Tree Feature Importances (mean across outer folds) =====")
for fs in feature_sets.keys():
    tmp = imp_mean_df[imp_mean_df["Feature_Set"] == fs]
    print(f"\n--- {fs} ---")
    print(tmp[["Feature", "Importance"]].to_string(index=False))

# Save outputs
foldwise_df.to_csv("decision_tree_foldwise.csv", index=False)
summary_df.to_csv("decision_tree_summary.csv", index=False)
imp_mean_df.to_csv("decision_tree_importances.csv", index=False)

# Plots
plot_metric_visuals(
    foldwise_df=foldwise_df,
    title_prefix="Decision Tree",
    out_prefix="decision_tree"
)

# Feature-importance bar plots (one per feature set)
for fs in feature_sets.keys():
    tmp = imp_mean_df[imp_mean_df["Feature_Set"] == fs].copy()
    tmp = tmp.sort_values("Importance")

    plt.figure()
    plt.barh(tmp["Feature"], tmp["Importance"])
    plt.title(f"Decision Tree Mean Feature Importance | {fs}")
    plt.xlabel("Mean Importance (outer folds)")
    plt.tight_layout()
    plt.savefig(f"decision_tree_importance_{fs.replace(' ', '_').replace('+','plus')}.png", dpi=300)
    plt.show()
